In [ ]:
!pip3 install pandas
!pip3 install seaborn
!pip3 install scikit-learn
!pip3 install matplotlib
!pip3 install numpy

import pandas as pd


In [ ]:
data = pd.read_csv('database.csv')

In [ ]:
missing_age = data['age'].isna().sum()
print(missing_age)




In [ ]:
import matplotlib.pyplot as plt

# Построение гистограммы
data['pclass'].value_counts().plot(kind='bar')
plt.xlabel('Класс')
plt.ylabel('Количество пассажиров')
plt.title('Распределение пассажиров по классам')
plt.show()


In [ ]:
survived_ratio = data['survived'].mean()
print(round(survived_ratio, 3))


In [ ]:
missing_ratios = data.isna().mean()
columns_to_drop = missing_ratios[missing_ratios > 0.33].index
data = data.drop(columns=columns_to_drop)

# Также удаляем колонку 'ticket'
data = data.drop(columns=['ticket'])


In [ ]:
data['fam_size'] = data['sibsp'] + data['parch']
data = data.drop(columns=['sibsp', 'parch'])
fam_size_mean = data['fam_size'].mean()
print(round(fam_size_mean, 3))


In [ ]:
num_predictors = data.drop(columns=['survived']).shape[1]
print(num_predictors)

In [ ]:
female_pclass1_survived = data[(data['sex'] == 'female') & (data['pclass'] == 1)]['survived'].mean()
print(round(female_pclass1_survived, 3))


In [ ]:
# Построение гистограммы для выживших
data[data['survived'] == 1]['age'].plot(kind='hist', bins=20, alpha=0.5, color='blue', label='Выжившие')

# Построение гистограммы для невыживших
data[data['survived'] == 0]['age'].plot(kind='hist', bins=20, alpha=0.5, color='red', label='Не выжившие')

plt.xlabel('Возраст')
plt.ylabel('Количество пассажиров')
plt.title('Распределение пассажиров по возрасту и статусу выживания')
plt.legend()
plt.show()


In [ ]:
numeric_data = data.select_dtypes(include=['float64', 'int64'])
numeric_data = numeric_data.dropna()

In [ ]:
from sklearn.model_selection import train_test_split

X = numeric_data.drop(columns=['survived'])
y = numeric_data['survived']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=33, stratify=y)


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

model = LogisticRegression(random_state=33, max_iter=1000)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
f1 = f1_score(y_test, y_pred)
print(round(f1, 3))


In [ ]:
numeric_data_filled = data.select_dtypes(include=['float64', 'int64'])
numeric_data_filled = numeric_data_filled.fillna(numeric_data_filled.mean())

In [ ]:
X = numeric_data_filled.drop(columns=['survived'])
y = numeric_data_filled['survived']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=33, stratify=y)

model = LogisticRegression(random_state=33, max_iter=1000)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
f1_filled = f1_score(y_test, y_pred)
print(round(f1_filled, 3))


In [ ]:
data['honorific'] = data['name'].str.extract(' ([A-Za-z]+)\.', expand=False)
unique_honorifics = data['honorific'].nunique()
print(unique_honorifics)


In [ ]:
data['honorific'] = data['honorific'].replace({
    'Rev': 'Mr', 'Col': 'Mr', 'Dr': 'Mr', 'Major': 'Mr', 'Don': 'Mr', 'Mme': 'Mrs', 'Dona': 'Mrs', 'Countess': 'Mrs',
    'Capt': 'Mr', 'Mlle': 'Miss', 'Ms': 'Miss'
})


In [ ]:
master_fraction = len(data[data['honorific'] == 'Master']) / len(data[data['sex'] == 'male'])
print(round(master_fraction, 3))

master_avg_age = data[data['honorific'] == 'Master']['age'].mean()
print(round(master_avg_age, 3))


In [ ]:
data['age'] = data.groupby('honorific')['age'].transform(lambda x: x.fillna(x.mean()))


In [ ]:
numeric_data = data.select_dtypes(include=['float64', 'int64']).drop(columns=['survived'])
y = data['survived']

X_train, X_test, y_train, y_test = train_test_split(numeric_data, y, test_size=0.2, random_state=33, stratify=y)

model = LogisticRegression(random_state=33, max_iter=1000)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
f1_honorific = f1_score(y_test, y_pred)
print(round(f1_honorific, 3))


In [ ]:
# Создаем колонку 'honorific' на основе имен
data['honorific'] = data['name'].str.extract(' ([A-Za-z]+)\.', expand=False)

# Агрегирование обращений
data['honorific'] = data['honorific'].replace({
    'Rev': 'Mr', 'Col': 'Mr', 'Dr': 'Mr', 'Major': 'Mr', 'Don': 'Mr', 'Mme': 'Mrs', 'Dona': 'Mrs', 'Countess': 'Mrs',
    'Capt': 'Mr', 'Mlle': 'Miss', 'Ms': 'Miss'
})

# Заполняем пропуски в колонке age средним значением для каждого обращения
data['age'] = data.groupby('honorific')['age'].transform(lambda x: x.fillna(x.mean()))


In [ ]:
data = data.drop(columns=['name', 'honorific'])
data = pd.get_dummies(data, drop_first=True)


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

# Разделение данных
X = data.drop(columns=['survived'])
y = data['survived']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=33, stratify=y)

# Обучение модели
model = LogisticRegression(random_state=42, max_iter=1000)
model.fit(X_train, y_train)

# Оценка модели
y_pred = model.predict(X_test)
f1 = f1_score(y_test, y_pred)
print(round(f1, 3))


In [ ]:
new_data = pd.read_csv('database2.csv')

# Создаем колонку 'honorific' на основе имен
new_data['honorific'] = new_data['name'].str.extract(' ([A-Za-z]+)\.', expand=False)

# Агрегирование обращений
new_data['honorific'] = new_data['honorific'].replace({
    'Rev': 'Mr', 'Col': 'Mr', 'Dr': 'Mr', 'Major': 'Mr', 'Don': 'Mr', 'Mme': 'Mrs', 'Dona': 'Mrs', 'Countess': 'Mrs',
    'Capt': 'Mr', 'Mlle': 'Miss', 'Ms': 'Miss'
})

# Заполняем пропуски в колонке age средним значением для каждого обращения
new_data['age'] = new_data.groupby('honorific')['age'].transform(lambda x: x.fillna(x.mean()))

new_data = new_data.drop(columns=['name', 'honorific'])
new_data = pd.get_dummies(new_data, drop_first=True)

new_data['fam_size'] = new_data['sibsp'] + new_data['parch']
new_data = new_data.drop(columns=['sibsp', 'parch'])

# Удаление ненужных колонок и выбор только числовых признаков
numeric_new_data = new_data.select_dtypes(include=['float64', 'int64'])


In [ ]:
X = data.drop(columns=['survived', 'sex_male', 'embarked_Q', 'embarked_S'])
y = data['survived']

print(numeric_new_data, X)

model = LogisticRegression(max_iter=1000)
model.fit(X, y)

new_predictions = model.predict(numeric_new_data)


In [ ]:
print(new_predictions)